# Repeated Days 1-3 benchmark runs (core + horizon sweep, x3)

Runs the same `core` and `horizon_sweep` experiment matrices from
`docs/DAYS_1_3_SPEC.md` three times consecutively, each repetition using the
identical set of seeds, to directly measure how much the results vary between
independent repetitions of the *same* nominal conditions.

With the core/horizon_sweep reuse fix (see below), each repetition executes
222 episodes (not 258 -- the 36 `h=10` rows shared with `core` are reused,
not re-run). At the measured average per-episode wall-clock time, 3
repetitions is roughly **~13.5 hours** total; adjust `NUM_REPEATS` below if
you want more or fewer.

This reuses the actual benchmark package (`async_vla_benchmark`) — the same
code path used for the Modal runs — rather than reimplementing anything, so
results here are directly comparable to prior runs.

## Prerequisites (must already be true in this Jupyter kernel's environment)

- `lerobot` installed with the `[pi,libero]` extras, ideally pinned to the same
  commit used for the Modal runs: `2aba372b4e217cc47db28e0f836859b20d1456c9`
  (see `modal_app.py`'s `LEROOT_COMMIT`). A different commit can silently
  change native latency and even success rates — see `docs/DECISIONS.md` /
  the "LEROBOT_COMMIT" discussion in this project's history.
- `torch` with CUDA available, `mujoco`, `robosuite`, `libero` (`hf-libero`).
- Task selection already run: `async_vla_benchmark/outputs/summaries/selected_tasks.json`
  must exist (from `async_vla_benchmark/scripts/select_tasks.py`). If you're
  running this against a fresh checkout, either copy that file over or run
  task selection first.
- A CUDA GPU. Model loading requires `device="cuda"` per `days1_3.yaml`.

## What this does *not* change

- The episode-level RNG seeding fix (`torch.manual_seed(seed)` in
  `EpisodeRunner.run()`) and the pinned `LEROBOT_COMMIT` are both already
  baked into the package/Modal image — this notebook doesn't need to redo
  either, it just needs to run in an environment that matches them.

## Core/horizon-sweep overlap

`core` and `horizon_sweep` both include `fixed_horizon=10` for
`naive_async`/`rtc`. Those are literally the same episode (same task, seed,
strategy, profile, horizon), and running the same episode twice would both
waste GPU time on this already-long sweep and silently overwrite the raw
`episodes/`/`requests`/`actions` files (both write to the same
`episode_id`-keyed paths). `run_tagged_plans` below reuses `core`'s result
for those 36-per-repetition rows instead of re-executing them.


In [ ]:
import sys
import json
import time
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "async_vla_benchmark").exists():
    # Adjust this if the notebook is opened from a different working directory.
    REPO_ROOT = Path.home() / "async-vla-latency-bench"
sys.path.insert(0, str(REPO_ROOT))

from async_vla_benchmark.benchmark.config import load_config
from async_vla_benchmark.benchmark.environment import get_task_info, make_libero_env
from async_vla_benchmark.benchmark.execution import run_episode
from async_vla_benchmark.benchmark.latency import LatencyProfile
from async_vla_benchmark.benchmark.logging import ensure_dir, write_json
from async_vla_benchmark.benchmark.policy import load_pi05_policy, load_pre_post_processors
from async_vla_benchmark.scripts.run_benchmark import (
    core_plans,
    horizon_plans,
    _episode_id,
    _parse_task,
    _selected_tasks_from_file,
)

print("repo root:", REPO_ROOT)
print("imports ok")


In [ ]:
# ---- Parameters ----

NUM_REPEATS = 3

CONFIG_PATH = REPO_ROOT / "async_vla_benchmark" / "configs" / "days1_3.yaml"
SELECTED_TASKS_FILE = REPO_ROOT / "async_vla_benchmark" / "outputs" / "summaries" / "selected_tasks.json"
BASE_OUTPUT_DIR = REPO_ROOT / "async_vla_benchmark" / "outputs" / "jupyter_repeats"

# Matches DAYS_1_3_SPEC.md: core uses 5 seeds, the horizon sweep uses 3.
# Every repetition reuses these same seed lists on purpose -- the point is to
# see how much the *same* nominal (task, strategy, profile, horizon, seed)
# condition varies across independent repetitions, not to sample new seeds.
CORE_SEEDS = [0, 1, 2, 3, 4]
HORIZON_SEEDS = [0, 1, 2]

# Skip episodes that already have a written episode_id.json from a prior
# partial run of this notebook (e.g. kernel restarted mid-way).
RESUME = True

print(f"will run {NUM_REPEATS} repetitions of core + horizon_sweep")
print(f"output base: {BASE_OUTPUT_DIR}")


In [ ]:
cfg = load_config(CONFIG_PATH)

tasks = _selected_tasks_from_file(SELECTED_TASKS_FILE)
if not tasks:
    raise RuntimeError(
        f"no selected tasks found at {SELECTED_TASKS_FILE}; "
        "run async_vla_benchmark/scripts/select_tasks.py first, or copy over "
        "an existing selected_tasks.json"
    )
print("selected tasks:", tasks)
print("policy checkpoint:", cfg.policy_checkpoint, "revision:", cfg.checkpoint_revision)


In [ ]:
# Load the policy exactly once. Every repetition and both experiments reuse
# this same loaded policy/preprocessor/postprocessor and the same env_cache,
# so the whole notebook run behaves like one continuous session -- the same
# principle used for the Modal `--experiment all` combined runs, which is
# what eliminated the cross-run GPU/latency confound seen when experiments
# were dispatched as separate containers.
print("loading policy (once, reused across all repetitions)...")
t0 = time.time()
policy = load_pi05_policy(
    cfg.policy_checkpoint,
    cfg.checkpoint_revision,
    n_action_steps=cfg.policy_n_action_steps,
    device=cfg.device,
)
preprocessor, postprocessor = load_pre_post_processors(
    policy, cfg.policy_checkpoint, cfg.checkpoint_revision
)
print(f"policy loaded in {time.time() - t0:.1f}s")

env_cache = {}


In [ ]:
def run_tagged_plans(tagged_plans, output_dir, resume=True):
    """Run every (experiment_name, plan) pair against output_dir.

    Mirrors run_benchmark.py's _run_experiment loop, but reuses the
    already-loaded policy/preprocessor/postprocessor and env_cache from the
    outer notebook session instead of loading them fresh. Writes one
    {experiment_name}_summaries.json per experiment represented in
    tagged_plans, under output_dir/summaries/.

    horizon_sweep's h=10 rows are, for shared (task, seed, strategy,
    profile), the same episode core_plans already covers (spec DAYS_1_3_SPEC.md
    section 16: "may reuse validated runs from the core experiment"). This
    reuses core's already-computed result for those instead of re-running an
    identical episode -- re-running it would waste GPU time on this already
    long (5x) sweep, and would silently overwrite core's raw
    episodes/requests/actions files, since both write to the same
    episode_id-keyed paths.
    """
    ensure_dir(output_dir / "requests")
    ensure_dir(output_dir / "actions")
    ensure_dir(output_dir / "episodes")
    ensure_dir(output_dir / "summaries")

    summaries_by_experiment = {}
    core_summaries_by_episode_id = {}
    for experiment_name, plan in tagged_plans:
        episode_id = _episode_id(plan)
        episode_json = output_dir / "episodes" / f"{episode_id}.json"

        if experiment_name == "horizon_sweep" and plan.fixed_horizon == 10:
            reused = core_summaries_by_episode_id.get(episode_id)
            if reused is None and episode_json.exists():
                reused = json.loads(episode_json.read_text())
            if reused is not None:
                summaries_by_experiment.setdefault(experiment_name, []).append(reused)
                print(f"reused {episode_id} from core: success={reused['success']} [{experiment_name}]")
                continue

        if resume and episode_json.exists():
            print(f"skip {episode_id}: already completed [{experiment_name}]")
            summaries_by_experiment.setdefault(experiment_name, []).append(
                json.loads(episode_json.read_text())
            )
            continue

        suite, task_id = _parse_task(plan.task)
        env_key = f"{suite}:{task_id}"
        if env_key not in env_cache:
            env_cache[env_key] = make_libero_env(
                suite,
                task_id,
                seed=plan.seed,
                control_mode=cfg.control_mode,
                obs_type=cfg.obs_type,
                camera_name=cfg.camera_name,
                observation_width=cfg.observation_width,
                observation_height=cfg.observation_height,
                init_states=cfg.init_states,
                episode_length=cfg.episode_length,
                num_steps_wait=cfg.num_steps_wait,
            )
        env = env_cache[env_key]

        latency_profile = next(
            (p for p in cfg.latency_profiles if p.name == plan.latency_profile), None
        )
        if latency_profile is None:
            raise ValueError(f"unknown latency profile {plan.latency_profile}")
        profile = LatencyProfile(
            latency_profile.name,
            latency_profile.use_measured_native_latency,
            latency_profile.added_latency_ms,
        )

        task_info = get_task_info(env, suite, task_id)
        summary = run_episode(
            env=env,
            policy=policy,
            preprocessor=preprocessor,
            postprocessor=postprocessor,
            task_instruction=task_info.language_instruction,
            episode_id=episode_id,
            strategy=plan.strategy,
            latency_profile=profile,
            fixed_horizon=plan.fixed_horizon,
            output_dir=output_dir,
            seed=plan.seed,
            use_rtc=(plan.strategy == "rtc"),
            rtc_execution_horizon=cfg.rtc.execution_horizon,
            request_threshold_actions=cfg.rtc.request_threshold_actions,
            device=cfg.device,
        )
        summaries_by_experiment.setdefault(experiment_name, []).append(summary)
        if experiment_name == "core":
            core_summaries_by_episode_id[episode_id] = summary
        print(
            f"completed {episode_id}: success={summary['success']} "
            f"steps={summary['environment_steps']} [{experiment_name}]"
        )

    for experiment_name, summaries in summaries_by_experiment.items():
        write_json(output_dir / "summaries" / f"{experiment_name}_summaries.json", summaries)
    return summaries_by_experiment


In [ ]:
results_by_repeat = {}

for rep in range(1, NUM_REPEATS + 1):
    print(f"\n{'='*30} REPETITION {rep}/{NUM_REPEATS} {'='*30}")
    rep_output_dir = BASE_OUTPUT_DIR / f"rep{rep}"
    tagged_plans = [("core", p) for p in core_plans(tasks, CORE_SEEDS)] + [
        ("horizon_sweep", p) for p in horizon_plans(tasks, HORIZON_SEEDS)
    ]
    print(f"planned episodes this repetition: {len(tagged_plans)}")
    t0 = time.time()
    results_by_repeat[rep] = run_tagged_plans(tagged_plans, rep_output_dir, resume=RESUME)
    print(f"repetition {rep} finished in {(time.time() - t0)/60:.1f} min")

print("\nall repetitions complete")


## Aggregate across repetitions

Loads every repetition's `core_summaries.json` / `horizon_sweep_summaries.json`,
tags each row with its `repetition` number, and combines them into two
DataFrames covering all `NUM_REPEATS` runs.


In [ ]:
import pandas as pd

all_core_rows = []
all_horizon_rows = []

for rep in range(1, NUM_REPEATS + 1):
    rep_summaries_dir = BASE_OUTPUT_DIR / f"rep{rep}" / "summaries"
    core_path = rep_summaries_dir / "core_summaries.json"
    horizon_path = rep_summaries_dir / "horizon_sweep_summaries.json"
    if core_path.exists():
        for row in json.loads(core_path.read_text()):
            row["repetition"] = rep
            all_core_rows.append(row)
    if horizon_path.exists():
        for row in json.loads(horizon_path.read_text()):
            row["repetition"] = rep
            all_horizon_rows.append(row)

core_df = pd.DataFrame(all_core_rows)
horizon_df = pd.DataFrame(all_horizon_rows)

combined_dir = BASE_OUTPUT_DIR / "combined"
combined_dir.mkdir(parents=True, exist_ok=True)
core_df.to_csv(combined_dir / "core_all_repeats.csv", index=False)
horizon_df.to_csv(combined_dir / "horizon_sweep_all_repeats.csv", index=False)

print("core_df:", core_df.shape, "-> saved to", combined_dir / "core_all_repeats.csv")
print("horizon_df:", horizon_df.shape, "-> saved to", combined_dir / "horizon_sweep_all_repeats.csv")


## Cross-repetition variance

For each (strategy, latency_profile) condition, this computes the success
rate *within* each repetition (averaged over that repetition's seeds), then
the mean and standard deviation of those five per-repetition success rates.
`success_rate_std_across_repeats` is a direct, empirical measure of how much
a full rerun of the same experiment moves the numbers -- exactly what earlier
single-run comparisons could only estimate indirectly.


In [ ]:
core_variance = (
    core_df.groupby(["strategy", "latency_profile", "repetition"])["success"]
    .mean()
    .reset_index()
    .groupby(["strategy", "latency_profile"])["success"]
    .agg(["mean", "std", "count"])
    .rename(columns={"mean": "success_rate_mean", "std": "success_rate_std_across_repeats"})
    .sort_index()
)
core_variance


In [ ]:
horizon_variance = (
    horizon_df.groupby(["strategy", "fixed_horizon", "latency_profile", "repetition"])["success"]
    .mean()
    .reset_index()
    .groupby(["strategy", "fixed_horizon", "latency_profile"])["success"]
    .agg(["mean", "std", "count"])
    .rename(columns={"mean": "success_rate_mean", "std": "success_rate_std_across_repeats"})
    .sort_index()
)
horizon_variance
